❌
not using this file anymore, but keeping it for reference
end up not using ocr and use the name list of NY state datasetinstaed


In [20]:
!pip install easyocr


  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached torch-2.6.0-cp312-none-macosx_11_0_arm64.whl.metadata (28 kB)
  Using cached torchvision-0.21.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.1 kB)
  Using cached opencv_python_headless-4.11.0.86-cp37-abi3-macosx_13_0_arm64.whl.metadata (20 kB)
  Using cached scipy-1.15.2-cp312-cp312-macosx_14_0_arm64.whl.metadata (61 kB)
  Using cached numpy-2.2.4-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pillow-11.1.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (9.1 kB)
  Using cached scikit_image-0.25.2-cp312-cp312-macosx_12_0_arm64.whl.metadata (14 kB)
  Using cached python_bidi-0.6.6-cp312-cp312-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached PyYAML-6.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.1 kB)
  Using cached shapely-2.1.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached pyclipper-1.3.0.post6-cp312-cp312-macosx_10_13_universal2.whl.metadata (9.0 kB)
  Using cach

In [1]:
import os
import cv2
import shutil
import pandas as pd
from tqdm.notebook import tqdm
import easyocr

In [ ]:
# Set paths
input_dir = "imgs/RICHMOND"
output_dir = "imgs/RICHMOND_cropped"
csv_path = "sign_detection_fast_results.csv"

# Initialize EasyOCR reader
reader = easyocr.Reader(['en'])

results = []

# Process each image
for filename in tqdm(os.listdir(input_dir)):
    if filename.lower().endswith(('.jpg')):
        full_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        try:
            img = cv2.imread(full_path)
            detections = reader.readtext(full_path)

            if not detections:
                results.append({"name": filename, "has_sign": "no"})
                continue

            # Compute bounding box that wraps all text
            x_min = min([min([p[0][0] for p in box]) for box, _, _ in detections])
            y_min = min([min([p[0][1] for p in box]) for box, _, _ in detections])
            x_max = max([max([p[0][0] for p in box]) for box, _, _ in detections])
            y_max = max([max([p[0][1] for p in box]) for box, _, _ in detections])

            # Crop and save
            cropped = img[int(y_min):int(y_max), int(x_min):int(x_max)]
            if cropped.size > 0:
                cv2.imwrite(output_path, cropped)
                results.append({"name": filename, "has_sign": "yes"})
            else:
                results.append({"name": filename, "has_sign": "no"})

        except Exception as e:
            results.append({"name": filename, "has_sign": "error"})

# Save results
df = pd.DataFrame(results)
df.to_csv(csv_path, index=False)
df.head()  # Show first few results


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

  0%|          | 0/361 [00:00<?, ?it/s]

,name,has_sign
0,704147.jpg,error
1,739577.jpg,error
2,750700.jpg,error
3,750066.jpg,error
4,641157.jpg,error


In [ ]:
import onnxruntime as ort
import numpy as np

# Load model
session = ort.InferenceSession("yolow-l.onnx")

# Example prompt and image
prompt = "store sign"
image_path = "imgs/RICHMOND/640032.jpg"

# Preprocess image
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (640, 640))
img_input = img_resized.astype(np.float32) / 255.0
img_input = np.transpose(img_input, (2, 0, 1))[np.newaxis, :]  # (1, 3, 640, 640)

# Encode text prompt (YOLO-World usually expects pre-tokenized prompt, but depends on model setup)
# ⚠️ If your ONNX model includes CLIP text encoding inside, we can send the prompt directly
# Otherwise, you'd need to encode prompt with CLIP and feed it

# Dummy text embedding if required — for now, assuming model takes only image input
inputs = {"images": img_input}

# Run inference
outputs = session.run(None, inputs)

print("Raw model outputs:", outputs)


2025-04-10 17:22:15.860429 [W:onnxruntime:, graph.cc:113 MergeShapeInfo] Error merging shape info for output. '/Gather_22_output_0' source:{-1,4} target:{}. Falling back to lenient merge.
2025-04-10 17:22:15.860885 [W:onnxruntime:, graph.cc:113 MergeShapeInfo] Error merging shape info for output. '/Gather_19_output_0' source:{-1,1} target:{}. Falling back to lenient merge.
2025-04-10 17:22:53.776633 [E:onnxruntime:, sequential_executor.cc:572 ExecuteKernel] Non-zero status code returned while running NonMaxSuppression node. Name:'/NonMaxSuppression' Status Message: non_max_suppression.cc:91 PrepareCompute boxes and scores should have same spatial_dimension.


Fail: [ONNXRuntimeError] : 1 : FAIL : Non-zero status code returned while running NonMaxSuppression node. Name:'/NonMaxSuppression' Status Message: non_max_suppression.cc:91 PrepareCompute boxes and scores should have same spatial_dimension.